# Retrieval Augmented Generation - Lab

Welcome to Week 6 Lab of AI Hands-On!

In this lab, we will:

* Set up our environment using standard RAG libraries
* Parse different types of files
* Compare three different chunking strategies
* Retrieve information that an LLM would not have normally access to
* Generate answers for user queries based on the retrieved information
* Evaluate our generated answers

To begin, run the following cell in order to install dependencies and get going:

In [1]:
import sys
import os

os.system("pip install -r requirements.txt")

0

## 1. Required Libraries
You must install the following Python packages which form the core of the RAG pipeline:

* langchain & langchain-community: The primary framework for building the application.

* langchain-openai: Integration for OpenAI's language models and embeddings.

* langchain-chroma: The vector database used for storing and retrieving text.

* pypdf & unstructured: Necessary for parsing PDF files and raw text.

* langchain-experimental: Required for advanced semantic chunking strategies.

* transformers & langchain-huggingface: Required if you intend to run models locally.

## 2. Model Dependencies
This lab uses two different sets of models depending on your access to API keys:

* Option A: Cloud Mode (OpenAI)

    * LLM: gpt-4o-mini.

    * Embeddings: OpenAIEmbeddings.

    * Requirement: An environment variable named OPENAI_API_KEY must be set.

* Option B: Local Mode (HuggingFace)

    * LLM: Qwen/Qwen2.5-0.5B.

    * Embeddings: sentence-transformers/all-MiniLM-L6-v2.

    * Requirement: Sufficient disk space to download these models via the transformers library.

## 3. Data Requirements

Data Folders: The script expects a directory structure containing your documents at ./rag_lab_data/pdf_docs and ./rag_lab_data/markdown_docs.

Python Version: The notebook is configured for Python 3.12.7.

In [2]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Check for API Key, if OPENAI_API_KEY is not set, we will use local models
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    api_key = os.getenv("OPENAI_API_KEY")

USE_LOCAL = not api_key

if USE_LOCAL:
    print("LOCAL MODE: Initializing Qwen2.5-0.5B + all-MiniLM-L6-v2")
    # Embedder: all-MiniLM-L6-v2 (384 dimensions)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    # Generator: Qwen2.5-0.5B
    model_id = "Qwen/Qwen2.5-0.5B"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    
    hf_pipe = pipeline(
        "text-generation", 
        model=model, 
        tokenizer=tokenizer, 
        max_new_tokens=128,
        return_full_text=False,
        temperature=0.1,
        max_length=None
    )
    llm = HuggingFacePipeline(pipeline=hf_pipe)
else:
    print("CLOUD MODE: Initializing OpenAI GPT-4o-mini...")
    embeddings = OpenAIEmbeddings()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # It is important to note that temperature=0 is set for cloud models to ensure deterministic outputs for evaluation purposes. For small LLMs we set 0.1 as is needed from the model we use in our lab

print("Initialization complete.")

CLOUD MODE: Initializing OpenAI GPT-4o-mini...
Initialization complete.


# 1. RAG Phase 1: RAG Ingestion

# Parsing

We begin by bringing the documents we want to Ingest in our environment.

This simple `route_and_parse` function which accepts PDF and Markdown files can easily be expanded to support more file formats by following the same logic.

Notice that if we try to ingest a file with an unsupported format the system will skip it.

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader

pdf_folder = "./rag_lab_data/pdf_docs"
markdown_folder = "./rag_lab_data/markdown_docs"

def route_and_parse(file_path):
    """Routes a single file to the correct loader based on its extension."""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        print(f"Routing PDF: {os.path.basename(file_path)}")
        loader = PyPDFLoader(file_path)
        return loader.load()

    elif ext == ".md":
        print(f"Routing MD: {os.path.basename(file_path)}")
        loader = TextLoader(file_path)
        return loader.load()

    else:
        print(f"Skipping unsupported file: {file_path}")
        return []

all_docs = []

# Parse all PDF files
print("Starting PDF Parsing")
for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        full_path = os.path.join(pdf_folder, filename)
        all_docs.extend(route_and_parse(full_path))

# Parse all Markdown files
print("Starting Markdown Parsing")
for filename in os.listdir(markdown_folder):
    if filename.endswith(".md"):
        full_path = os.path.join(markdown_folder, filename)
        all_docs.extend(route_and_parse(full_path))

print(f"\nTotal documents loaded: {len(all_docs) -1}")

Starting PDF Parsing
Routing PDF: Builder AG Project.pdf
Starting Markdown Parsing
Routing MD: builder_ag_logs.md

Total documents loaded: 2


# Chunking Strategies

LLMs have context limits, and embedding models usually have even stricter limits (e.g., 384 tokens). We cannot feed a 100-page PDF into the database all at once. We must split it into smaller, manageable chunks.

We will compare three chunking strategies on a snippet of our text file:

* **Fixed-Size Chunking**: Split strictly by a set number of characters or tokens.

* **Recursive Chunking**: Split by paragraphs, then sentences, then words, only falling back to raw characters if necessary to respect the size limit.

* **Semantic Chunking**: Advanced AI chunking. Use embeddings to measure the meaning of sentences, grouping them together until the topic shifts.

In [4]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

sample_text = all_docs[0].page_content[:1500]

print("----------------------")
print("1. Fixed-Size Chunking")
print("----------------------")
# Splits strictly by character limit.
fixed_splitter = CharacterTextSplitter(separator="", chunk_size=300, chunk_overlap=30)
fixed_chunks = fixed_splitter.split_text(sample_text)
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"Chunk {i+1} (Length {len(chunk)}):\n{chunk}...\n")

print("----------------------")
print("2. Recursive Chunking")
print("----------------------")
# Splits on logical breaks (\n\n, \n, spaces) to keep sentences intact. Chunk overlap is used to maintain context across chunks. In this lab
# we set a 10% overlap for better context retention, but this can be adjusted based on the use case and model capabilities.
rec_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
rec_chunks = rec_splitter.split_text(sample_text)
for i, chunk in enumerate(rec_chunks[:3]):
    print(f"Chunk {i+1} (Length {len(chunk)}):\n{chunk}...\n")

print("----------------------")
print("3. Semantic Chunking")
print("----------------------")
# Uses an embedding model to group conceptually similar sentences.
semantic_splitter = SemanticChunker(embeddings, breakpoint_threshold_type="percentile")
semantic_chunks = semantic_splitter.split_text(sample_text)
for i, chunk in enumerate(semantic_chunks[:3]):
    print(f"Chunk {i+1} (Length {len(chunk)}):\n{chunk}...\n")

----------------------
1. Fixed-Size Chunking
----------------------
Chunk 1 (Length 300):
Builder  AG  Heavy  Civil  &  Marine  Constructors:  Internal  Logistics  &  
Engineering
 
Manifesto
 
Effective  Date:  March  2026  Confidentiality  Level:  Site-Command  Only  (Marine  &  
Subterranean)
 
1.  Corporate  Overview  and  Q3  2026  Objectives  
Builder  AG  Heavy  Civil  remains  th...

Chunk 2 (Length 299):
AG  Heavy  Civil  remains  the  premier  contractor  for  high-stress,  extreme-environment  
infrastructure.
 
As
 
we
 
transition
 
into
 
Q3
 
2026,
 
our
 
primary
 
corporate
 
directive
 
shifts
 
from
 
surface-level
 
commercial
 
builds
 
to
 
abyssal-zone
 
and
 
high-pressure
 
subterra...

Chunk 3 (Length 300):
and
 
high-pressure
 
subterranean
 
anchoring.
 
This
 
requires
 
a
 
frictionless
 
workflow
 
between
 
Marine
 
Logistics,
 
Subterranean
 
Engineering,
 
and
 
the
 
Materials
 
Science
 
divisions.
 
All
 
site
 
managers
 
must
 
ensure
 
crush-dep

# Vector Store & Indexing

After the chunking process, our data must be converted into vector embeddings and stored (and indexed) in a system designed for high-speed retrieval based on cosine similarity.

### 1. Vector Store Definition
A **Vector Store** is a database optimized for storing and querying high-dimensional numerical arrays. 
* **Embeddings:** Every text chunk is processed into a list of floating-point numbers (a vector) representing its semantic content.
* **Mathematical Proximity:** Retrieval is based on calculating the distance between vectors using metrics such as cosine similarity.
* **Metadata Storage:** The system stores the original text and its associated metadata alongside the vector for reference during retrieval.



---

### 2. Indexing and HNSW
**Indexing** is the computational organization of vectors to minimize search latency. In large datasets, a "Brute Force" search is computationally expensive.

* **HNSW (Hierarchical Navigable Small World):** This is a graph-based indexing algorithm.
* **Efficiency:** With HNSW data points are organized into a graph structure. 
* **Search Execution:** The algorithm traverses these layers to find the most relevant data points without performing a full database scan.



---

### 3. Technical Workflow

1.  **Embedding Generation:** A model transforms text chunks into high-dimensional vectors.
2.  **Indexing:** The vector store inserts these vectors into an HNSW graph structure.
3.  **Query Processing:** A user's question is converted into a vector using the same model.
4.  **Nearest Neighbor Search:** The system traverses the index to identify the vectors with the smallest mathematical distance to the query vector.

---

### Vector Database - ChromaDB (Used in this Lab)
We utilize **Chroma** (via `langchain-chroma`) for its specific implementation features:
* **Local Operations:** It runs entirely within the local environment, eliminating the need for external server infrastructure.
* **Automated Indexing:** It manages the HNSW and Approximate Nearest Neighbor (ANN) calculations automatically.
* **Persistence:** We preserve the index in a local directory, allowing data to be loaded across different sessions without re-embedding.

In [ ]:
import os
from langchain_chroma import Chroma

# Define a local directory for storage
CHROMA_PATH = "chroma_db"

print(f"Vector Storage")

current_collection = "builder_local" if USE_LOCAL else "builder_openai"

# Create the vector store
# This turns our chunked docs into numerical vectors and saves them to our disk
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name=current_collection,
    persist_directory=CHROMA_PATH
)

print(f"Vector Store created at '{CHROMA_PATH}'.")

Vector Storage
Vector Store created at 'chroma_db'.


# 2. RAG Phase 2: Retrieval & Generation

# Retrieval

When a user asks a question, we embed their question into the same mathematical space we have stored our documents and find the closest matching chunks using cosine similarity. 

---

### **Retrieval Techniques** (Dense, Sparse, & Hybrid)

In a basic RAG system, we rely purely on Dense Retrieval (Vector Embeddings). While powerful for capturing semantic meaning, this approach can struggle with exact keyword matches, names, or specific acronyms.

To improve our system, we combine multiple retrieval strategies:

* **Dense Retrieval (Vector Search)**: Uses AI embeddings to find documents with similar meaning.

* **Sparse Retrieval (Keyword Search)**: Uses algorithms (BM25) to look for exact word frequencies and matches. Does not understand meaning, but is useful for finding specific terms.

* **Hybrid Retrieval**: Combines the results of both Dense and Sparse retrieval. In this lab, Reciprocal Rank Fusion (RRF) is used to re-rank the combined results, utilising semantic understanding and keyword precision.

---

### **Reciprocal Rank Fusion (RRF) Algorithm**

RRF is an algorithm used to combine multiple search result lists into a single, optimized ranking.

RRF takes into account the **rank** (position) of the document in each list.

The formula we are going to use for implementing our Hybrid Retrieval Logic:

$$Score = \sum_{d \in D} \frac{1}{k + r(d)}$$

* **$r(d)$**: The rank of document $d$ in a specific list (e.g., is it the 1st result or the 5th?).
* **$k$**: A smoothing constant (usually set to **60**) that ensures highly ranked documents don't completely vanish if they appear lower in one of the lists.

### **The Logic in our Custom Class:**
1.  **Parallel Retrieval:** We trigger the **Dense** and **Sparse** retrievers simultaneously.
2.  **Scoring:** For every document found, we calculate its RRF score. If a document appears at **Rank 1** in both the Vector search AND the Keyword search, its score is boosted significantly.
3.  **Fusion:** We merge the lists provided by the two retrievers we initiated. Documents that appear in both lists are fused by summing their reciprocal rank scores.
4.  **Final Ranking:** We sort the entire collection by the new combined RRF score and pass the top results to the LLM.

**This ensures the LLM receives context that is both semantically relevant AND contains the exact technical keywords requested.**

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

print("Retrieval Techniques")

# Initialize the Dense Retriever directly from the vector store
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Dense Retriever linked to ChromaDB.")

ingested_data = vectorstore.get(include=['documents', 'metadatas'])
ingested_docs = [
    Document(page_content=doc, metadata=meta) 
    for doc, meta in zip(ingested_data['documents'], ingested_data['metadatas'])
]

sparse_retriever = BM25Retriever.from_documents(ingested_docs)
sparse_retriever.k = 2
print(f"Sparse Retriever initialized using {len(ingested_docs)} documents from ChromaDB.")

# Reciprocal Rank Fusion
class CustomHybridRetriever:
    def __init__(self, dense, sparse):
        self.dense = dense
        self.sparse = sparse

    def invoke(self, query):
        dense_docs = self.dense.invoke(query)
        sparse_docs = self.sparse.invoke(query)
        
        rrf_scores = {}
        all_docs_map = {}
        
        # RRF: Score = 1 / (Rank + 60)
        for rank, doc in enumerate(dense_docs):
            doc_id = doc.page_content
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 / (rank + 60))
            all_docs_map[doc_id] = doc
            
        for rank, doc in enumerate(sparse_docs):
            doc_id = doc.page_content
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 / (rank + 60))
            all_docs_map[doc_id] = doc
            
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [all_docs_map[doc_id] for doc_id, score in sorted_docs[:2]]

hybrid_retriever = CustomHybridRetriever(dense_retriever, sparse_retriever)
print("Hybrid Retriever (RRF) synchronized with ChromaDB.")

Retrieval Techniques
Dense Retriever linked to ChromaDB.
Sparse Retriever initialized using 27 documents from ChromaDB.
Hybrid Retriever (RRF) synchronized with ChromaDB.


## **Evaluating Retrieval: Dense, Sparse, and Hybrid**

Before an LLM can generate an answer, the **Retriever** must find the right information. We evaluate this by checking if the ground truth appears in the top results.

### **Why Evaluate Retrieval Separately?**
* **Identify Hallucinations:** If the retriever fails, the LLM will "guess" or say it doesn't know. 
* **Optimize Chunking:** If the retrieved text is cut off in the middle of a protocol, the LLM can't give a complete answer.
* **Benchmark Accuracy:** We use **Hit Rate** to see if the correct document made it into the top results.

In [7]:
# Ground Truth Dataset
# Each 'must_have' is a list of words that prove the right information was found.
eval_cases = [
    {"q": "Who is the Chief Architect?", "must_have": ["Elena", "Rostova"]},
    {"q": "What is HVT-77?", "must_have": ["HVT", "77"]},
    {"q": "What happened on March 18th?", "must_have": ["2.4", "amber"]},
    {"q": "What happened on March 21st?", "must_have": ["scaffolding", "joint"]},
    {"q": "Penalty for lost equipment?", "must_have": ["docking", "pay"]},
    {"q": "Show me the specific log entry for March 15.", "must_have": ["Hyperbaric Chamber 4", "suits cleared"]}
]

retrievers = {
    "Dense": dense_retriever,
    "Sparse": sparse_retriever,
    "Hybrid": hybrid_retriever
}

# Initialize counters
scores = {name: 0 for name in retrievers.keys()}

print(f"{'Question':<30} | {'Dense':<12} | {'Sparse':<12} | {'Hybrid':<12}")
print("-" * 75)

for case in eval_cases:
    row = f"{case['q']:<30} | "
    for name, r in retrievers.items():
        docs = r.invoke(case['q'])
        combined_content = " ".join([d.page_content.lower() for d in docs])
        
        is_hit = all(k.lower() in combined_content for k in case['must_have'])
        
        if is_hit:
            scores[name] += 1
            status = "HIT"
        else:
            status = "NO HIT"
            
        source = os.path.basename(docs[0].metadata.get('source', '???')) if docs else "N/A"
        row += f"{status} ({source[:7]:<7}) | "
    print(row)

print("-" * 75)
print("Scores:")
for name, count in scores.items():
    print(f"{name:<7}: {count}/{len(eval_cases)} ({int(count/len(eval_cases)*100)}%)")

Question                       | Dense        | Sparse       | Hybrid      
---------------------------------------------------------------------------
Who is the Chief Architect?    | HIT (Builder) | HIT (Builder) | HIT (Builder) | 
What is HVT-77?                | HIT (builder) | HIT (Builder) | HIT (builder) | 
What happened on March 18th?   | HIT (builder) | HIT (builder) | HIT (builder) | 
What happened on March 21st?   | HIT (builder) | HIT (builder) | HIT (builder) | 
Penalty for lost equipment?    | NO HIT (Builder) | HIT (builder) | HIT (Builder) | 
Show me the specific log entry for March 15. | HIT (builder) | HIT (builder) | HIT (builder) | 
---------------------------------------------------------------------------
Scores:
Dense  : 5/6 (83%)
Sparse : 6/6 (100%)
Hybrid : 6/6 (100%)


# Generation

As a final step, we pass the retrieved context along with our original query, to the LLM.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import re

# Define the User Question
user_question = "Who is the Chief Architect of the Bering Strait project and what is the protocol if a cryogenetic thermal sign is detected?"

# Retrieval Step (Defining 'context_text')
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retrieved_docs = hybrid_retriever.invoke(user_question)
context_text = format_docs(retrieved_docs)

# Define the two template options
standard_template = """You are a technical assistant for Builder AG. 
Use ONLY the following retrieved context to answer the question.
If the answer is not in the context, say "I don't know based on the provided context."

Context:
{context}

Question: {question}

Answer:"""

# template for local models
local_template = """Use the following context to answer the question.
If the answer is not in context, say "I don't know based on the provided context."
Context:{context}
Question: {question}
Answer:"""

if USE_LOCAL:
    prompt = ChatPromptTemplate.from_template(local_template)
    augmented_input = prompt.format(context=context_text, question=user_question)
else:
    prompt = ChatPromptTemplate.from_template(standard_template)
    augmented_input = prompt.format_messages(context=context_text, question=user_question)

ai_response_raw = llm.invoke(augmented_input)


if hasattr(ai_response_raw, 'content'):
    response = ai_response_raw.content
else:
    # Note: small LLMs often require cleaning of their output for the final answer
    response = re.sub(r'(?i)assistant\s*:', '', str(ai_response_raw)).strip()
    response = response.split("Question:")[0].split("Context:")[0].strip()

print(f"--- AI Response using ({'Local Qwen' if USE_LOCAL else 'OpenAI GPT'}) ---")
print("User Question:", user_question)
print("Response:", response)

--- AI Response using (OpenAI GPT) ---
User Question: Who is the Chief Architect of the Bering Strait project and what is the protocol if a cryogenetic thermal sign is detected?
Response: The Chief Architect of the Bering Strait project is Elena Rostova. If a cryogenic thermal sign, specifically an amber glow, is detected on the staging deck, personnel must immediately initiate the Flash-Quench sequence to flood the chamber with liquid nitrogen.


---
We will also compare the response of our RAG system against a "Vanilla" response (a response of the same LLM without the use of RAG)

In [9]:
test_questions = [
    "Who is Elena Rostova?",
    "What is HVT-77 and how should it be stored?",
    "Give me a summary of the incident on March 18, 2026."
]

def generate_answer(llm, question):
    """Generates a response using only the model's internal knowledge."""
    if USE_LOCAL:
        prompt = f"{question}"
    else:
        prompt = question
        
    raw_res = llm.invoke(prompt)
    
    # Extract and clean text
    text = raw_res.content if hasattr(raw_res, 'content') else str(raw_res)
    return re.sub(r'(?i)assistant\s*:', '', text).strip()

# RAG
def answer_with_rag(question, llm, retriever):
    """Retrieves context first, then generates a grounded response."""
    # Retrieval
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    
    # Augmentation
    if USE_LOCAL:
        prompt_input = local_template.format(context=context, question=question)
    else:
        prompt_input = ChatPromptTemplate.from_template(standard_template).format_messages(
            context=context, question=question
        )
        
    # Generation
    raw_res = llm.invoke(prompt_input)
    
    # Cleanup
    text = raw_res.content if hasattr(raw_res, 'content') else str(raw_res)
    clean_ans = re.sub(r'(?i)assistant\s*:', '', text).strip()
    
    return clean_ans, docs

print(f"--- Running Comparison ({'Local' if USE_LOCAL else 'OpenAI'}) ---\n")

# Vanilla answer (no RAG)
for test_question in test_questions:
    print(f"\n{'='*80}")
    vanilla_ans = generate_answer(llm, test_question)
    print("QUESTION:", test_question)
    print("VANILLA:", vanilla_ans)
    print()
# RAG answer
    rag_ans, retrieved = answer_with_rag(test_question, llm, hybrid_retriever)
    print("RAG:", rag_ans)
    print()

print("--- Retrieved passages ---")
for i, r in enumerate(retrieved, 1):
    source = r.metadata.get('source', 'Unknown').split('/')[-1]
    print(f"\n[{i}] Source: {source}")
    print(f"    {r.page_content[:150]}...")

--- Running Comparison (OpenAI) ---


QUESTION: Who is Elena Rostova?
VANILLA: Elena Rostova is a fictional character from Leo Tolstoy's novel "War and Peace," which was first published in the 1860s. She is portrayed as a beautiful and ambitious woman, often associated with the aristocratic society of Russia during the Napoleonic Wars. Elena is the wife of Count Pierre Bezukhov, one of the novel's central characters, and her character embodies themes of vanity, social ambition, and moral ambiguity. Her relationships and actions throughout the story reflect the complexities of love, power, and the societal expectations of women in her time. If you are looking for more specific information or analysis about her character, feel free to ask!

RAG: Elena Rostova is the Chief Architect overseeing the Project Zenith at the Bering Strait site. She assumed command of the site in January 2026 and is responsible for the direct supervision of the entire Zenith initiative, which involves the comple

## **Evaluating RAG: The LLM-as-a-Judge Pattern**

Testing a RAG system is difficult because traditional code tests are too rigid. *Specific example from this lab: An AI might answer "The project is led by Elena Rostova,"* which is factually correct but would fail a standard string-matching test. 

To solve this, we use **LLM-as-a-Judge**.

---

### **How the Evaluation Pipeline Works**

We treat a second LLM instance (the "Judge") as an impartial auditor that compares three distinct pieces of data:

1.  **The Ground Truth:** The real answer manually derived from our project files.
2.  **The Generated Answer:** What our RAG pipeline actually produced.
3.  **The Retrieved Context:** The specific chunks of text the retriever handed to the LLM.


### **The Three Aspects of the Grade**

The Judge assigns a score (0.0 to 1.0) by analyzing these specific relationships:

* **Faithfulness:** Did the AI answer using *only* the provided context, or did it hallucinate?
* **Answer Relevance:** Does the answer actually address the user's question?
* **Context Precision:** Did the retriever find the exact sentences needed to support the ground truth?


In [16]:
import json
import re

def extract_json_safely(raw_input):
    """
    Helper function to accurately extract the information that we need from the model's output, even if it's not perfectly formatted JSON.
    """
    text = raw_input.content if hasattr(raw_input, 'content') else str(raw_input)
    
    text = re.split(r'(?i)assistant\s*:|answer\s*:', text)[-1].strip()
    text = re.sub(r'```json|```', '', text).strip()
    
    try:
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1:
            data = json.loads(text[start:end+1])
            return {
                "score": float(data.get('score', 0.0)),
                "reason": data.get('reason', 'Parsed successfully.')
            }
        
        score_match = re.search(r'(?i)score[:\s]+([0-1]\.?\d*)', text)
        if score_match:
            return {"score": float(score_match.group(1)), "reason": "Regex score extraction."}
            
        number_only = re.search(r'^\s*([0-1]\.?\d*)\s*$', text)
        if number_only:
            return {"score": float(number_only.group(1)), "reason": "Direct numeric output."}

    except Exception:
        pass
    
    return {"score": 0.0, "reason": f"Failed to parse model output: {text[:50]}..."}
    
# Evaluation Dataset
eval_dataset = [
    {"question": "Who is the Chief Architect of Project Zenith?", "ground_truth": "Elena Rostova."},
    {"question": "What is the protocol for an amber thermal signature?", "ground_truth": "Immediately initiate the Flash-Quench sequence to flood the chamber with liquid nitrogen."},
    {"question": "What happened on March 18, 2026?", "ground_truth": "A cooling fluctuation led to a faint amber thermal signature on HVT-77 Canister 09; Flash-Quench was used."},
    {"question": "What is the drone activation time limit for the Mariana Protocol?", "ground_truth": "I don't know."}, # we intentionally set the ground_truth to something that is 
                                                                                                                        #   not in the context to test the judge so we would expect a score close to 0.0 for this question
    {"question": "What happened on March 21, 2026?", "ground_truth": "A scaffolding joint snapped; debris was captured in 42 seconds by automated submersibles."},
    {"question": "How often is structural telemetry submitted?", "ground_truth": "Every 12 hours to Chief Rostova's terminal."},
    {"question": "What happened to the relay network on March 24, 2026?", "ground_truth": "A 4-minute blackout occurred during a scheduled encryption update."},
    {"question": "How do you access Zenith structural schematics?", "ground_truth": "Biometric verification and physical presence in a shielded command module."},
    {"question": "When do hyperbaric seals need recalibration?", "ground_truth": "After every 50 decompression cycles."},
    {"question": "How much did the temperature rise on March 18?", "ground_truth": "2.4°C."}
]

# Standard Templates
standard_rag_temp = "Use context to answer: \nContext: {context}\nQuestion: {question}"

standard_judge_temp = """Grade the "Generated Answer" against the "Ground Truth". 
Score 0.0 to 1.0. Return ONLY JSON: {{"score": float, "reason": string}}
Ground Truth: {ground_truth} | Answer: {answer} | Context: {context}"""

# Because our local model is small, the same template will not work as intended, so we need to adjust the prompt to suit our smaller model

#Local Model RAG Template
local_rag_temp = """Use the following context to answer the question.
If the answer is not in context, say "I don't know based on the provided context."
Context:
{context}

Question: {question}
Answer:"""

# Local Model Judge Template
local_judge_temp = """### SYSTEM:
You are a strict grading auditor. Your ONLY job is to output a JSON object. 
DO NOT answer the user's question. DO NOT explain the protocol. 
ONLY compare the AI Answer to the Ground Truth.

### INPUT:
Ground Truth: {ground_truth}
AI Answer: {answer}

### OUTPUT FORMAT:
{{"score": 1.0, "reason": "Perfect match"}} 

(Use 1.0 for correct, 0.0 for wrong, or 0.5 for partial).
JSON ONLY:"""

# Evaluation Loop
print(f"--- Running Evaluation ({'Local' if USE_LOCAL else 'OpenAI'}) ---\n")

all_scores = []

for i, item in enumerate(eval_dataset, 1):
    # Retrieval
    docs = hybrid_retriever.invoke(item["question"])
    context_text = "\n".join([d.page_content for d in docs])
    
    # Generation (RAG Answer)
    if USE_LOCAL:
        rag_input = local_rag_temp.format(context=context_text, question=item["question"])
    else:
        rag_prompt = ChatPromptTemplate.from_template(standard_rag_temp)
        rag_input = rag_prompt.format_messages(context=context_text, question=item["question"])
    
    gen_raw = llm.invoke(rag_input)

    gen_answer = gen_raw.content if hasattr(gen_raw, 'content') else str(gen_raw)
    gen_answer = gen_answer.split("Question:")[0].strip()
    
    # Judging
    if USE_LOCAL:
        j_input = local_judge_temp.format(ground_truth=item["ground_truth"], answer=gen_answer)
    else:
        j_prompt = ChatPromptTemplate.from_template(standard_judge_temp)
        j_input = j_prompt.format_messages(ground_truth=item["ground_truth"], answer=gen_answer, context=context_text)
    
    judge_raw = llm.invoke(j_input)
    
    # Parsing of the final output & Scoring
    judge_res = extract_json_safely(judge_raw)
    score = judge_res.get('score', 0.0)
    all_scores.append(score)
    
    # Progress Logging
    print(f"[{i}/{len(eval_dataset)}] Q: {item['question']}")
    print(f"   Score: {score} | Reason: {judge_res.get('reason')}")
    print("-" * 30)

# Evaluation Results Summary
if all_scores:
    avg_score = (sum(all_scores) / len(all_scores)) * 100
    print(f"\n{'='*40}")
    print(f"FINAL ACCURACY: {avg_score:.1f}%")
    print(f"TOTAL EVALUATED: {len(all_scores)}")
    print(f"{'='*40}")

--- Running Evaluation (OpenAI) ---

[1/10] Q: Who is the Chief Architect of Project Zenith?
   Score: 1.0 | Reason: The generated answer correctly identifies Elena Rostova as the Chief Architect of Project Zenith, which matches the information provided in the ground truth.
------------------------------
[2/10] Q: What is the protocol for an amber thermal signature?
   Score: 0.9 | Reason: The generated answer closely follows the ground truth by stating the need to initiate the Flash-Quench sequence upon observing an amber thermal signature. It adds context about the volatile properties of Hyper-Viscous Thermo-Resin, which enhances the explanation but slightly diverges from the brevity of the ground truth. Overall, it maintains the essential information and urgency of the original instruction.
------------------------------
[3/10] Q: What happened on March 18, 2026?
   Score: 0.9 | Reason: The generated answer accurately captures the key details from the ground truth, including the dat